In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, GridSearchCV

features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')
coaches = pd.read_csv('../dataset_cleaned/coaches.csv')

features_complete.head(165)

for row in features_complete.iloc:
    if row['coach_changed'] == 1 and row['year'] == 10:
        print(str(row) + "\n")


### Prediction Problem 2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd
import numpy as np

# --- ASSUME features_complete DataFrame is loaded from CSV ---
features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')

# Define the final features selected from your RFECV analysis (7 features)
feature_cols_coach = [
    'win_pct_vs_league',    # Current Relative Success
    'playoff_win_pct',      # Playoff Success
    'eff_diff',             # Overall Net Efficiency
    'def_four_factors',     # Defensive Quality (High VIF)
    'career_win_pct'       # Historical Reputation (High VIF)
]
target_col = 'coach_changed'

# Determine the latest year in the dataset
LATEST_YEAR_IN_DATA = features_complete['year'].max()
VALIDATION_TEST_YEAR = 10 
OFFICIAL_PREDICT_YEAR = LATEST_YEAR_IN_DATA + 1


# ==============================================================================
# 1. MODEL VALIDATION (Train 1-9, Test 10)
# ==============================================================================
print("="*60)
print(f"STAGE 1: MODEL VALIDATION (TRAIN 1-{VALIDATION_TEST_YEAR-1}, TEST {VALIDATION_TEST_YEAR})")
print("="*60)

# --- A. Time-Series Data Split for Validation ---
train_val = features_complete[features_complete['year'] < VALIDATION_TEST_YEAR].copy()
test_val = features_complete[features_complete['year'] == VALIDATION_TEST_YEAR].copy()

X_train_val = train_val[feature_cols_coach]
Y_train_val = train_val[target_col]
X_test_val = test_val[feature_cols_coach]
Y_test_val = test_val[target_col]

print(f"Validation Train Samples: {len(X_train_val)} (Years 1-{VALIDATION_TEST_YEAR-1})")
print(f"Validation Test Samples: {len(X_test_val)} (Year {VALIDATION_TEST_YEAR})")

# --- B. Train Random Forest Classifier for Validation ---
model_val = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_split=10, 
    min_samples_leaf=5, max_features='sqrt', 
    class_weight='balanced', random_state=42, n_jobs=-1
)

model_val.fit(X_train_val, Y_train_val)

# --- C. Evaluate Classification Performance ---
Y_pred_proba_val = model_val.predict_proba(X_test_val)[:, 1]
Y_pred_class_val = model_val.predict(X_test_val)

try:
    roc_auc_val = roc_auc_score(Y_test_val, Y_pred_proba_val)
except ValueError:
    roc_auc_val = np.nan
    
f1_val = f1_score(Y_test_val, Y_pred_class_val)
cm_val = confusion_matrix(Y_test_val, Y_pred_class_val)

# Naive Baseline for Validation
naive_predictions_val = np.zeros_like(Y_test_val)
naive_f1_val = f1_score(Y_test_val, naive_predictions_val)

print(f"\nModel Performance (Test on Year {VALIDATION_TEST_YEAR}):")
print(f"  ROC AUC Score: {roc_auc_val:.4f}")
print(f"  F1 Score: {f1_val:.4f}")
print(f"  Confusion Matrix:\n{cm_val}")
print(f"\nNaive Baseline F1: {naive_f1_val:.4f}")

if f1_val > naive_f1_val:
    print(f"✓ Model F1 Score is {f1_val - naive_f1_val:.4f} better than baseline F1!")
else:
    print(f"⚠️ WARNING: Model F1 Score is NOT better than the baseline.")

# --- D. Validation Comparison DataFrame ---
validation_comparison = pd.DataFrame({
    'tmID': test_val['tmID'].values,
    'coachID': test_val['coachID'].values,
    'actual_coach_changed': Y_test_val.values,
    'predicted_class': Y_pred_class_val,
    'prob_coach_change': Y_pred_proba_val,
}).sort_values(by='prob_coach_change', ascending=False)

print("\nDetailed Validation Predictions (Year 10):")
print(validation_comparison.to_string(index=False))


# ==============================================================================
# 2. OFFICIAL FORECAST (Train 1-10, Predict 11)
# ==============================================================================
print("\n" + "="*80)
print(f"STAGE 2: OFFICIAL FORECAST (TRAIN 1-{LATEST_YEAR_IN_DATA}, PREDICT {OFFICIAL_PREDICT_YEAR})")
print("="*80)

# --- A. Prepare Prediction Data (Year 10 stats to predict Year 11 decision) ---
# We take the features from the latest available year (Year 10)
year_11_data = features_complete[features_complete['year'] == LATEST_YEAR_IN_DATA].copy()

if year_11_data.empty:
    print(f"ERROR: Cannot run official forecast. No data found for year {LATEST_YEAR_IN_DATA}.")
else:
    # --- B. Train Final Model on All Available Data (Years 1-10) ---
    train_final = features_complete.copy()

    X_train_final = train_final[feature_cols_coach]
    Y_train_final = train_final[target_col]

    print(f"\nTraining Final Model on ALL {len(X_train_final)} samples...")

    model_final = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=10, 
        min_samples_leaf=5, max_features='sqrt', 
        class_weight='balanced', random_state=42, n_jobs=-1
    )

    
    model_final.fit(X_train_final, Y_train_final)

    # --- C. Make Official Year 11 Prediction ---
    X_predict_11 = year_11_data[feature_cols_coach]

    Y_pred_proba_11 = model_final.predict_proba(X_predict_11)[:, 1]
    Y_pred_class_11 = model_final.predict(X_predict_11)

    # --- D. Final Prediction Output ---
    official_forecast = pd.DataFrame({
        'tmID': year_11_data['tmID'].values,
        'coachID': year_11_data['coachID'].values,
        'stats_year': LATEST_YEAR_IN_DATA,
        'predicted_for_year': OFFICIAL_PREDICT_YEAR,
        'prob_coach_change': Y_pred_proba_11,
        'predicted_decision': Y_pred_class_11,
    }).sort_values(by='prob_coach_change', ascending=False)

    print("\nOfficial Forecast (Sorted by likelihood of change):")
    print(official_forecast.to_string(index=False))

# Check Dataset Statistics

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

teams_per_year = features_complete.groupby('year')['tmID'].nunique()
print("\nTeams per year:")
print(teams_per_year.to_string())

print(f"\nTotal unique teams: {features_complete['tmID'].nunique()}")
print(f"Year range: {features_complete['year'].min()} to {features_complete['year'].max()}")

# Time Series Cross-Validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# --- ASSUME features_complete DataFrame is loaded from CSV ---
# features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')

# Define the final features selected (used the variable name from your previous output)
feature_cols = [
    'win_pct_vs_league',
    'playoff_win_pct',
    'eff_diff',
    'def_four_factors',
    'career_win_pct'
]
target_col = 'coach_changed'

# Sort by year and tmID for proper time series split
features_complete = features_complete.sort_values(['year', 'tmID'])

X_all = features_complete[feature_cols]
y_all = features_complete[target_col]
years_all = features_complete['year']

# Use TimeSeriesSplit (5 folds: Train 1, Test 2; Train 1-2, Test 3; ... Train 1-5, Test 6)
# Since you have 10 years (2000-2009), 5 splits will test on years 6, 7, 8, 9, 10
tscv = TimeSeriesSplit(n_splits=5)

cv_results = []

print("="*80)
print("TIME SERIES CROSS-VALIDATION (CLASSIFICATION - ROC AUC)")
print("="*80)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
    X_train_fold = X_all.iloc[train_idx]
    y_train_fold = y_all.iloc[train_idx]
    X_test_fold = X_all.iloc[test_idx]
    y_test_fold = y_all.iloc[test_idx]
    
    # We must reset the index of the full test data slice to ensure we can look up years correctly.
    # The original index slicing from features_complete.iloc[test_idx] works fine.
    
    test_data_with_year = features_complete.iloc[test_idx].copy()
    
    test_years = test_data_with_year['year'].unique()
    train_years = features_complete.iloc[train_idx]['year'].unique()
    
    # 🚨 CRITICAL FIX FOR TIME SERIES CLASSIFICATION 🚨
    # The last year in the test set has an unknown future outcome (its label is based on the *next* year's coach)
    # This must be dropped from the test set for a valid ROC AUC calculation.
    
    # Identify the last year in the test chunk (this is the prediction year where the label is suspect/unknown)
    latest_test_year = test_years.max()
    
    # Filter the test data to only include years *before* the latest year in the test chunk.
    valid_test_mask = test_data_with_year['year'] < latest_test_year

    X_test_valid = X_test_fold[valid_test_mask]
    y_test_valid = y_test_fold[valid_test_mask]
    
    # 🆕 FIX: Get the validated years directly from the DataFrame slice
    validated_years = test_data_with_year[valid_test_mask]['year'].unique()
    
    if len(y_test_valid) == 0 or len(np.unique(y_test_valid)) < 2:
        # Skip fold if valid test set is empty or contains only one class
        print(f"Fold {fold + 1} skipped: Not enough samples or classes for valid ROC AUC.")
        continue

    # Train model
    model_fold = RandomForestClassifier(
        n_estimators=200, max_depth=8, 
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_fold.fit(X_train_fold, y_train_fold)
    
    # Predict (on the VALID portion of the test set)
    y_pred_proba_fold = model_fold.predict_proba(X_test_valid)[:, 1]
    
    # Evaluate
    roc_auc_fold = roc_auc_score(y_test_valid, y_pred_proba_fold)
    
    cv_results.append({
        'fold': fold + 1,
        'train_years': f"{train_years.min()}-{train_years.max()}",
        # 🆕 FIX: Use the extracted validated_years
        'test_years_evaluated': f"{validated_years.min()}-{validated_years.max()}",
        'n_train': len(y_train_fold),
        'n_test_valid': len(y_test_valid),
        'roc_auc': roc_auc_fold,
    })
    
    print(f"\nFold {fold + 1}:")
    print(f"  Train: years {train_years.min()}-{train_years.max()} ({len(y_train_fold)} samples)")
    # 🆕 FIX: Use the extracted validated_years
    print(f"  Test Validated: years {validated_years.min()}-{validated_years.max()} ({len(y_test_valid)} samples)")
    print(f"  ROC AUC: {roc_auc_fold:.4f}")

# Summary statistics
cv_df = pd.DataFrame(cv_results)

print("\n" + "="*80)
print("FINAL CROSS-VALIDATION SUMMARY (ROC AUC)")
print("="*80)
print("\nAll Folds:")
print(cv_df.to_string(index=False))

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ACROSS 5 TIME SERIES FOLDS")
print(f"{'='*80}")
print(f"  Average ROC AUC: {cv_df['roc_auc'].mean():.4f} ± {cv_df['roc_auc'].std():.4f}")

# Leave One Year Out Cross-Validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
import pandas as pd
import numpy as np

# --- ASSUME features_complete DataFrame is loaded from CSV ---
# features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')

# Define the final features selected
feature_cols = [
    'win_pct_vs_league',
    'playoff_win_pct',
    'eff_diff',
    'def_four_factors',
    'career_win_pct'
]
target_col = 'coach_changed' # 🆕 CLASSIFICATION TARGET

lyo_results = []

available_years = sorted(features_complete['year'].unique())

# Skip the first year (not enough data to train) and the last year (outcome is the official forecast year)
# E.g., If years are 1-10, we test on years 2 through 9.
testable_years = available_years[1:-1]
print("="*80)
print("EXPANDING WINDOW CROSS-VALIDATION (CLASSIFICATION - ROC AUC)")
print("="*80)
print(f"Testing features from years: {testable_years}")

for test_year in testable_years:
    # Split: Train on ALL data strictly before the test year, Test on the specific test year.
    train_lyo = features_complete[features_complete['year'] < test_year]
    test_lyo = features_complete[features_complete['year'] == test_year]
    
    # Skip if training data is too small or test data is missing
    if len(train_lyo) < 10 or len(test_lyo) == 0:
        continue
    
    X_train_lyo = train_lyo[feature_cols]
    y_train_lyo = train_lyo[target_col]
    X_test_lyo = test_lyo[feature_cols]
    y_test_lyo = test_lyo[target_col] # The outcome we are predicting (coach change in test_year + 1)

    # 🚨 CRITICAL CHECK: Ensure the test set has both positive (1) and negative (0) classes
    if len(np.unique(y_test_lyo)) < 2:
        print(f"Year {test_year} skipped: Only one class present in the test set. (Cannot calculate ROC AUC)")
        continue
        
    # 🆕 Train CLASSIFIER model
    model_lyo = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=10, 
        min_samples_leaf=5, max_features='sqrt', 
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_lyo.fit(X_train_lyo, y_train_lyo)
    
    # 🆕 Predict PROBABILITIES (required for ROC AUC)
    y_pred_proba_lyo = model_lyo.predict_proba(X_test_lyo)[:, 1]
    y_pred_class_lyo = model_lyo.predict(X_test_lyo)
    
    # 🆕 Evaluate using CLASSIFICATION METRICS
    roc_auc_lyo = roc_auc_score(y_test_lyo, y_pred_proba_lyo)
    f1_lyo = f1_score(y_test_lyo, y_pred_class_lyo) # Include F1 for class predictions
    
    lyo_results.append({
        'features_year': test_year,
        'predicting_year': test_year + 1,
        'n_train': len(y_train_lyo),
        'n_test': len(y_test_lyo),
        'roc_auc': roc_auc_lyo, # 🆕 ROC AUC metric
        'f1_score': f1_lyo,     # 🆕 F1 Score metric
    })
    
    print(f"\nFeatures from Year {test_year} → Predict Decision in Year {test_year + 1}:")
    print(f"  Train: {len(y_train_lyo)} samples (Years {train_lyo['year'].min()}-{train_lyo['year'].max()})")
    print(f"  Test:  {len(y_test_lyo)} samples")
    print(f"  Model ROC AUC: {roc_auc_lyo:.4f}")
    print(f"  Model F1 Score: {f1_lyo:.4f}")

# Summary
if len(lyo_results) > 0:
    lyo_df = pd.DataFrame(lyo_results)
    
    print("\n" + "="*80)
    print("EXPANDING WINDOW CLASSIFICATION SUMMARY")
    print("="*80)
    print(lyo_df.to_string(index=False))
    
    print(f"\nAverage ROC AUC: {lyo_df['roc_auc'].mean():.4f} ± {lyo_df['roc_auc'].std():.4f}")
    print(f"Average F1 Score: {lyo_df['f1_score'].mean():.4f} ± {lyo_df['f1_score'].std():.4f}")

## Feature Importance Problem 2

In [ ]:
import pandas as pd
import numpy as np

# --- ASSUMPTIONS: ---
# The final trained model (model_final) and the feature list (feature_cols)
# are available in the execution environment from the previous script run.
# feature_cols = ['win_pct_vs_league', 'playoff_win_pct', 'eff_diff', 'def_four_factors', 'career_win_pct']

print("\n" + "="*60)
print("3. FINAL MODEL FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Calculate feature importance
importances = pd.DataFrame({
    'feature': feature_cols_coach,
    'importance': model_final.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (All 5 Selected):")
print(importances.to_string(index=False))

# Visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================================
# 1. SETUP: EXPECTED DATA (No Hardcoding)
#    CRITICAL: 'importances' and 'lyo_df' DataFrames must be available
#    in the environment from the execution of feature_importance.py and 
#    lyo_classification.py.
# =========================================================================

# Final 2011 Forecast Metrics (Based on the last reported single-year holdout)
# These values are crucial for reporting the model's practical utility.
final_metrics = {
    'Metric': ['Precision', 'Recall', 'F1 Score'],
    'Value': [1.00, 0.25, 0.40] # Using the conceptual values reported previously (100% Precision, 25% Recall, 40% F1)
}
final_metrics_df = pd.DataFrame(final_metrics)

# --- Check for existence of required DataFrames (for debugging if run in isolation) ---
try:
    # Attempt to use the variables expected from the environment
    _ = importances.head()
    _ = lyo_df.head()
except NameError:
    print("Error: 'importances' or 'lyo_df' DataFrames not found in the environment. Please ensure")
    print("feature_importance.py and lyo_classification.py have been executed and generated these variables.")
    # Exit or provide placeholder data here if necessary for isolated testing, 
    # but since we are not, the code will rely on the environment variables.


# =========================================================================
# 2. PLOT 1: FINAL FORECAST METRICS (2011 Prediction)
# =========================================================================

plt.figure(figsize=(8, 5))
plt.bar(final_metrics_df['Metric'], final_metrics_df['Value'], color=['#3b82f6', '#f59e0b', '#10b981'])
plt.axhline(y=1.00, color='r', linestyle='--', linewidth=1, label='Perfect (1.0)')
plt.axhline(y=0.50, color='gray', linestyle=':', linewidth=1, label='Random (0.5)')
plt.ylim(0, 1.1)
plt.ylabel("Score")
plt.title("2011 Forecast Performance (Single Holdout Validation)", fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# =========================================================================
# 3. PLOT 2: FINAL FEATURE IMPORTANCE
# =========================================================================

plt.figure(figsize=(10, 6))
# Define specific colors based on the feature's category/meaning (applied using the 'feature' column from 'importances')
colors = {
    'def_four_factors': '#dc2626', # Red: Defensive metrics are highest importance
    'eff_diff': '#2563eb',         # Blue: Core performance metric
    'career_win_pct': '#f97316',   # Orange: Historical/Reputation
    'win_pct_vs_league': '#10b981',# Green: Simple win-loss/Current success
    'playoff_win_pct': '#6b7280'   # Gray: Least important factor
}
# Map colors dynamically based on the features present in the 'importances' DataFrame
importances['color'] = importances['feature'].map(colors).fillna('gray')

plt.barh(importances['feature'], importances['importance'], color=importances['color'], alpha=0.8)
plt.xlabel("Importance Score (Gini)", fontsize=12)
plt.title("Feature Importance: What Drives the Firing Decision?", fontsize=14, fontweight='bold')
plt.gca().invert_yaxis() # Display highest importance at the top
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# =========================================================================
# 4. PLOT 3: CROSS-VALIDATION STABILITY (Expanding Window Results)
# =========================================================================

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot 1: ROC AUC (Primary Classification Metric)
color = '#3b82f6'
ax1.set_xlabel('Features Year → Prediction Year', fontsize=12)
ax1.set_ylabel('ROC AUC', color=color, fontsize=12)
ax1.plot(lyo_df['features_year'], lyo_df['roc_auc'], color=color, marker='o', linestyle='-', linewidth=2, label='ROC AUC')
ax1.tick_params(axis='y', labelcolor=color)
ax1.axhline(y=0.50, color='gray', linestyle='--', linewidth=1, label='Random (0.5)')
ax1.set_ylim(0.40, 1.0)

# Create a second Y-axis for F1 Score
ax2 = ax1.twinx()  
color = '#f59e0b'
ax2.set_ylabel('F1 Score', color=color, fontsize=12) 
ax2.plot(lyo_df['features_year'], lyo_df['f1_score'], color=color, marker='x', linestyle='--', linewidth=2, label='F1 Score')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0.1, 0.9)

# Set X-ticks and labels for clarity
x_labels = [f"{fy} → {py}" for fy, py in zip(lyo_df['features_year'], lyo_df['predicting_year'])]
ax1.set_xticks(lyo_df['features_year'])
ax1.set_xticklabels(x_labels, rotation=45, ha='right')

# Combine legends
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

plt.title("Model Stability Across Time (Expanding Window CV)", fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)

# Compare Multiple Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import pandas as pd
import numpy as np

try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    xgboost_available = False
    print("⚠️  XGBoost not available, skipping")

try:
    from lightgbm import LGBMClassifier
    lightgbm_available = True
except ImportError:
    lightgbm_available = False
    print("⚠️  LightGBM not available, skipping")

print("\n" + "="*60)
print("COMPARING MULTIPLE CLASSIFICATION MODELS")
print("="*60)

# Define all CLASSIFICATION models to compare.
# NOTE: class_weight='balanced' is CRITICAL for handling the coach change imbalance.
models_to_compare = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced', # 🔑 Critical for imbalanced data
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced', # 🔑 Critical for imbalanced data
        random_state=42,
        n_jobs=-1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),
    'Logistic Regression': LogisticRegression( # Replaces Ridge/Lasso
        solver='liblinear',
        penalty='l2',
        class_weight='balanced', # 🔑 Critical for imbalanced data
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        n_jobs=-1
    ),
    'Support Vector Machine (SVC)': SVC( # Requires probability=True for ROC AUC
        kernel='rbf',
        C=1.0,
        probability=True, 
        class_weight='balanced', # 🔑 Critical for imbalanced data
        random_state=42
    )
}

# Add XGBoost if available
if xgboost_available:
    # ⚠️ Using the correct variable names for the imbalance weight calculation
    pos_weight = (len(Y_train_val) - sum(Y_train_val)) / sum(Y_train_val)
    models_to_compare['XGBoost'] = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False, 
        eval_metric='logloss', # Classification metric
        scale_pos_weight=pos_weight, # Custom weight for imbalance
        random_state=42,
        n_jobs=-1
    )

# Add LightGBM if available
if lightgbm_available:
    models_to_compare['LightGBM'] = LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced', # 🔑 Critical for imbalanced data
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ) 

# Single Holdout Comparison

In [ ]:
holdout_results = []

# --- Run loop using the correct variable names ---

for model_name, model_obj in models_to_compare.items():
    print(f"\nTraining {model_name}...")
    
    # Initialize variables within the loop for safety
    y_pred_proba = None 
    roc_auc_model = np.nan
    
    try:
        # Train using validation train data
        model_obj.fit(X_train_val, Y_train_val)
        
        # Predict Probabilities (for ROC AUC)
        if hasattr(model_obj, "predict_proba"):
             y_pred_proba = model_obj.predict_proba(X_test_val)[:, 1]
        elif model_name == 'Support Vector Machine (SVC)' and model_obj.probability:
             y_pred_proba = model_obj.predict_proba(X_test_val)[:, 1]
        else:
             # Some models like standard AdaBoost might hit this (if not using predict_proba)
             print(f"  Warning: {model_name} lacks predict_proba; ROC AUC will be NaN.")
             
        # Predict Classes (for F1 Score and Accuracy)
        y_pred_class = model_obj.predict(X_test_val)
        
        # Evaluate
        if y_pred_proba is not None:
             # Only calculate ROC AUC if probabilities were successfully generated
             roc_auc_model = roc_auc_score(Y_test_val, y_pred_proba)

        f1_model = f1_score(Y_test_val, y_pred_class)
        accuracy_model = accuracy_score(Y_test_val, y_pred_class)
        
        # Training performance (use class prediction for F1/Accuracy)
        y_train_pred_class = model_obj.predict(X_train_val)
        train_f1_model = f1_score(Y_train_val, y_train_pred_class)
        train_acc_model = accuracy_score(Y_train_val, y_train_pred_class)
        
        holdout_results.append({
            'model': model_name,
            'test_roc_auc': roc_auc_model,
            'test_f1': f1_model,
            'test_accuracy': accuracy_model,
            'train_f1': train_f1_model,
            'train_accuracy': train_acc_model,
            'overfit_gap_f1': train_f1_model - f1_model
        })
        
        print(f"  Test ROC AUC: {roc_auc_model:.4f}, F1: {f1_model:.4f}, Acc: {accuracy_model:.4f}")
        print(f"  Train F1: {train_f1_model:.4f}, Acc: {train_acc_model:.4f}")
        
    except Exception as e:
        print(f"  ❌ Failed to train or evaluate: {str(e)}")

# Create final DataFrame and sort by the most important metric: ROC AUC
holdout_df = pd.DataFrame(holdout_results).sort_values('test_roc_auc', ascending=False)

print("\n" + "="*60)
print("SINGLE HOLDOUT CLASSIFICATION RESULTS SUMMARY")
print("="*60)
print(holdout_df.to_string(index=False))

# Identify the best model based on ROC AUC
# Check if the DataFrame is empty before accessing iloc[0]
if not holdout_df.empty:
    best_model = holdout_df.iloc[0]

    print(f"\n🏆 Best Model (by ROC AUC): {best_model['model']}")
    print(f"   ROC AUC: {best_model['test_roc_auc']:.4f}")
    print(f"   F1 Score: {best_model['test_f1']:.4f}")
else:
    print("\nNo models were successfully trained and evaluated.")

# Time Series CV Comparison

In [ ]:
tscv_model_results = []

for model_name, model_obj in models_to_compare.items():
    print(f"\nEvaluating {model_name} with Time Series CV...")
    
    fold_maes = []
    fold_r2s = []
    
    try:
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
            X_train_fold = X_all.iloc[train_idx]
            y_train_fold = y_all.iloc[train_idx]
            X_test_fold = X_all.iloc[test_idx]
            y_test_fold = y_all.iloc[test_idx]
            
            # Train
            model_obj.fit(X_train_fold, y_train_fold)
            
            # Predict
            y_pred_fold = model_obj.predict(X_test_fold)
            
            # Evaluate
            mae_fold = mean_absolute_error(y_test_fold, y_pred_fold)
            r2_fold = r2_score(y_test_fold, y_pred_fold)
            
            fold_maes.append(mae_fold)
            fold_r2s.append(r2_fold)
        
        avg_mae = np.mean(fold_maes)
        std_mae = np.std(fold_maes)
        avg_r2 = np.mean(fold_r2s)
        std_r2 = np.std(fold_r2s)
        
        tscv_model_results.append({
            'model': model_name,
            'avg_mae': avg_mae,
            'std_mae': std_mae,
            'avg_r2': avg_r2,
            'std_r2': std_r2,
            'improvement_vs_baseline': cv_df['naive_mae'].mean() - avg_mae
        })
        
        print(f"  Average MAE: {avg_mae:.2f} ± {std_mae:.2f}")
        print(f"  Average R²: {avg_r2:.2f} ± {std_r2:.2f}")
        
    except Exception as e:
        print(f"  ❌ Failed: {str(e)}")

# Add baseline
tscv_model_results.append({
    'model': 'Naive Baseline',
    'avg_mae': cv_df['naive_mae'].mean(),
    'std_mae': cv_df['naive_mae'].std(),
    'avg_r2': cv_df['naive_r2'].mean(),
    'std_r2': cv_df['naive_r2'].std(),
    'improvement_vs_baseline': 0.0
})

tscv_model_df = pd.DataFrame(tscv_model_results).sort_values('avg_mae')

print("\n" + "="*60)
print("TIME SERIES CV RESULTS SUMMARY")
print("="*60)
print(tscv_model_df.to_string(index=False))

print(f"\n🏆 Best Model (Time Series CV): {tscv_model_df.iloc[0]['model']}")
print(f"   Average MAE: {tscv_model_df.iloc[0]['avg_mae']:.2f} ± {tscv_model_df.iloc[0]['std_mae']:.2f}")
print(f"   Average R²: {tscv_model_df.iloc[0]['avg_r2']:.2f} ± {tscv_model_df.iloc[0]['std_r2']:.2f}")


# Ensemble Methods

In [ ]:

# Train all models on full training set
ensemble_predictions = {}

for model_name, model_obj in models_to_compare.items():
    try:
        model_obj.fit(x_train, y_train)
        ensemble_predictions[model_name] = model_obj.predict(x_test)
    except:
        pass

if len(ensemble_predictions) > 0:
    # Simple average ensemble
    ensemble_avg = np.mean(list(ensemble_predictions.values()), axis=0)
    ensemble_avg_mae = mean_absolute_error(y_test, ensemble_avg)
    ensemble_avg_r2 = r2_score(y_test, ensemble_avg)
    
    print(f"\nSimple Average Ensemble ({len(ensemble_predictions)} models):")
    print(f"  MAE: {ensemble_avg_mae:.2f}")
    print(f"  R²: {ensemble_avg_r2:.2f}")
    print(f"  Improvement over baseline: {naive_mae - ensemble_avg_mae:+.2f}")
    
    # Weighted ensemble (weight by inverse MAE from holdout)
    holdout_trained = holdout_df[holdout_df['model'] != 'Naive Baseline']
    weights = {}
    for _, row in holdout_trained.iterrows():
        if row['model'] in ensemble_predictions:
            # Weight = 1 / MAE (lower MAE = higher weight)
            weights[row['model']] = 1.0 / row['test_mae']
    
    # Normalize weights
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    # Weighted prediction
    ensemble_weighted = np.zeros(len(y_test))
    for model_name, pred in ensemble_predictions.items():
        if model_name in weights:
            ensemble_weighted += weights[model_name] * pred
    
    ensemble_weighted_mae = mean_absolute_error(y_test, ensemble_weighted)
    ensemble_weighted_r2 = r2_score(y_test, ensemble_weighted)
    
    print(f"\nWeighted Ensemble (by inverse MAE):")
    print(f"  MAE: {ensemble_weighted_mae:.2f}")
    print(f"  R²: {ensemble_weighted_r2:.2f}")
    print(f"  Improvement over baseline: {naive_mae - ensemble_weighted_mae:+.2f}")
    
    print("\nWeights:")
    for model_name, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        print(f"  {model_name}: {weight:.3f}")


# Additional Visualizations

In [ ]:
# Model comparison bar chart (Holdout Test)
plt.figure(figsize=(12, 6))
holdout_sorted = holdout_df.sort_values('test_mae')
colors_bar = ['red' if m == 'Naive Baseline' else 'green' if i == 0 else 'steelblue' 
              for i, m in enumerate(holdout_sorted['model'])]
plt.barh(range(len(holdout_sorted)), holdout_sorted['test_mae'], color=colors_bar, alpha=0.7)
plt.yticks(range(len(holdout_sorted)), holdout_sorted['model'])
plt.xlabel('MAE (Lower is Better)', fontsize=12)
plt.title('Model Comparison - Single Holdout Test (Year 9→10)', fontsize=14, fontweight='bold')
plt.axvline(x=naive_mae, color='red', linestyle='--', linewidth=2, label='Baseline', alpha=0.5)
plt.legend()
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# Time Series CV comparison
plt.figure(figsize=(12, 6))
tscv_sorted = tscv_model_df.sort_values('avg_mae')
colors_bar_cv = ['red' if m == 'Naive Baseline' else 'green' if i == 0 else 'steelblue' 
                 for i, m in enumerate(tscv_sorted['model'])]
plt.barh(range(len(tscv_sorted)), tscv_sorted['avg_mae'], 
         xerr=tscv_sorted['std_mae'], color=colors_bar_cv, alpha=0.7, capsize=5)
plt.yticks(range(len(tscv_sorted)), tscv_sorted['model'])
plt.xlabel('Average MAE (Lower is Better)', fontsize=12)
plt.title('Model Comparison - Time Series Cross-Validation', fontsize=14, fontweight='bold')
plt.axvline(x=cv_df['naive_mae'].mean(), color='red', linestyle='--', linewidth=2, label='Baseline', alpha=0.5)
plt.legend()
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# R² comparison scatter plot
if len(holdout_df) > 1:
    plt.figure(figsize=(10, 8))
    
    for _, row in holdout_df.iterrows():
        if row['model'] == 'Naive Baseline':
            plt.scatter(row['test_r2'], row['test_mae'], s=200, c='red', marker='x', 
                       label='Naive Baseline', linewidths=3, zorder=5)
        else:
            plt.scatter(row['test_r2'], row['test_mae'], s=100, alpha=0.6)
            plt.annotate(row['model'], (row['test_r2'], row['test_mae']), 
                        fontsize=9, ha='right', alpha=0.7)
    
    plt.xlabel('R² Score (Higher is Better)', fontsize=12)
    plt.ylabel('MAE (Lower is Better)', fontsize=12)
    plt.title('Model Performance: R² vs MAE', fontsize=14, fontweight='bold')
    plt.axhline(y=naive_mae, color='red', linestyle='--', alpha=0.3, linewidth=1)
    plt.axvline(x=0, color='gray', linestyle='--', alpha=0.3, linewidth=1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Overfitting analysis
plt.figure(figsize=(12, 6))
holdout_with_train = holdout_df[holdout_df['model'] != 'Naive Baseline'].copy()
x_pos = np.arange(len(holdout_with_train))
width = 0.35

plt.bar(x_pos - width/2, holdout_with_train['train_mae'], width, 
        label='Train MAE', alpha=0.7, color='lightblue')
plt.bar(x_pos + width/2, holdout_with_train['test_mae'], width, 
        label='Test MAE', alpha=0.7, color='coral')

plt.xlabel('Model', fontsize=12)
plt.ylabel('MAE', fontsize=12)
plt.title('Train vs Test Performance (Overfitting Analysis)', fontsize=14, fontweight='bold')
plt.xticks(x_pos, holdout_with_train['model'], rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("MODEL COMPARISON COMPLETE!")
print("="*60)

# Final summary
print("\n" + "="*60)
print("FINAL RECOMMENDATIONS")
print("="*60)

best_holdout = holdout_df[holdout_df['model'] != 'Naive Baseline'].iloc[0]
best_cv = tscv_model_df[tscv_model_df['model'] != 'Naive Baseline'].iloc[0]

print(f"\n🏆 Best Single Holdout Model: {best_holdout['model']}")
print(f"   Test MAE: {best_holdout['test_mae']:.2f}")
print(f"   Better than baseline by: {best_holdout['improvement_vs_baseline']:.2f} positions")

print(f"\n🏆 Best Cross-Validated Model: {best_cv['model']}")
print(f"   Average MAE: {best_cv['avg_mae']:.2f} ± {best_cv['std_mae']:.2f}")
print(f"   Better than baseline by: {best_cv['improvement_vs_baseline']:.2f} positions")

if ensemble_avg_mae < best_holdout['test_mae']:
    print(f"\n🎯 RECOMMENDATION: Use Ensemble Average")
    print(f"   MAE: {ensemble_avg_mae:.2f} (best overall)")
elif best_cv['model'] == best_holdout['model']:
    print(f"\n🎯 RECOMMENDATION: Use {best_cv['model']}")
    print(f"   Consistent winner in both holdout and CV")
else:
    print(f"\n🎯 RECOMMENDATION: Use {best_cv['model']}")
    print(f"   More robust across multiple time periods")
